In [ ]:
%pip install -U langchain-huggingface sentence-transformers pypdf python-docx ipywidgets


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os

In [ ]:
from io import BytesIO

import ipywidgets as widgets
from IPython.display import display
from docx import Document as WordDocument
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from pypdf import PdfReader

In [29]:
load_dotenv()


use_openai = os.getenv("USE_OPENAI", "false").strip().lower() == "true"

if not use_openai:
    raise RuntimeError("Set USE_OPENAI=true to initialize these embeddings.")

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("USE_OPENAI=true requires OPENAI_API_KEY in .env")
model = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
base_url = os.getenv("base_url")

embeddings = OpenAIEmbeddings(
    model=model,
    api_key=api_key,
    base_url=base_url,
)



In [ ]:
seed_documents = [
    Document(
        page_content="LangChain 是一個快速串接語言模型與外部工具的開源框架。",
        metadata={"source": "built-in"},
    ),
    Document(
        page_content="向量資料庫專門用於存取和檢索向量化資料，可實現語義搜尋。",
        metadata={"source": "built-in"},
    ),
    Document(
        page_content="RAG 結合檢索與生成能力，提供有根據的即時回答。",
        metadata={"source": "built-in"},
    ),
]


def _uploaded_items(upload_value):
    return upload_value.values() if isinstance(upload_value, dict) else upload_value


def _read_uploaded_file(uploaded_file):
    file_name = uploaded_file["name"]
    file_bytes = bytes(uploaded_file["content"])
    suffix = file_name.lower().rsplit(".", 1)[-1] if "." in file_name else ""

    if suffix in {"txt", "md", "csv", "json"}:
        return file_bytes.decode("utf-8", errors="replace")
    if suffix == "pdf":
        reader = PdfReader(BytesIO(file_bytes))
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    if suffix == "docx":
        word_document = WordDocument(BytesIO(file_bytes))
        return "\n".join(paragraph.text for paragraph in word_document.paragraphs)
    raise ValueError(f"Unsupported file type: .{suffix or 'unknown'}")


def _chunk_text(text, chunk_size=1200, overlap=200):
    text = " ".join(text.split())
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks


uploader = widgets.FileUpload(
    accept=".txt,.md,.csv,.json,.pdf,.docx",
    multiple=True,
    description="Choose files",
)
rebuild_button = widgets.Button(description="Build RAG index", button_style="primary")
status = widgets.Output()


def rebuild_index(_=None):
    global documents, vector_store
    documents = list(seed_documents)
    skipped_files = []
    for uploaded_file in _uploaded_items(uploader.value):
        try:
            text = _read_uploaded_file(uploaded_file)
            documents.extend(
                Document(page_content=chunk, metadata={"source": uploaded_file["name"]})
                for chunk in _chunk_text(text)
            )
        except ValueError as error:
            skipped_files.append(str(error))

    vector_store = InMemoryVectorStore.from_documents(documents, embeddings)
    with status:
        status.clear_output()
        print(f"Indexed {len(documents)} chunks from {len(uploader.value)} uploaded file(s).")
        for error in skipped_files:
            print(error)


rebuild_button.on_click(rebuild_index)
display(uploader, rebuild_button, status)
rebuild_index()

In [ ]:
# The index is built by the upload cell above. Re-run that cell after selecting files.

In [ ]:
query = "什麼是RAG？"
search_result = vector_store.similarity_search(query, k=3)

for index, doc in enumerate(search_result, start=1):
    print(f"[{index}] {doc.metadata.get('source', 'unknown')}\n{doc.page_content}\n")

RAG 結合檢索與生成能力，提供有根據的即時回答。
